In [1]:
from controller.siglent.siglent_analyzer import SiglentAnalyzer
import cv2



In [2]:
analyzer = SiglentAnalyzer("170.0.0.2", 5025)

Respuesta CHECKIDN: Siglent Technologies,SSA3015X Plus,SVA1XA2X900311,3.2.2.6.3R2


In [31]:
analyzer.set_center_freq(152)


In [30]:

cap = cv2.VideoCapture(0)
detector = cv2.QRCodeDetector()
while True:
    _, img = cap.read()

    data, bbox,_ = detector.detectAndDecode(img)
    if data:
        print(data)
        break

cap.release()

ULAD-B4 37201


# Configure ULAD

In [4]:
import serial
from legacy_bench_support.ulad_amplifier import UladAmplifier
dut_amplifier = UladAmplifier(serial_port='/dev/ttyACM0')

In [5]:
dl_agc = dut_amplifier.get_agc("DL")
print(dl_agc)


1025


In [49]:
dut_amplifier.set_agc("DL", 1025)

True

# Gain demo DL

In [6]:
# Set instrument to DL
analyzer.set_center_freq(470)
analyzer.set_span(0)
analyzer.set_marker(1, 470)
analyzer.set_resolution_bandwidth_khz(0.3)


In [7]:
# Enable TG
analyzer.set_tracking_generator_level(-20)
analyzer.set_tracking_generator("ON")

In [8]:
# Read values
analyzer.get_level_from_marker(1)

-14.06

In [10]:
TARGET_PW = -26.5
TOLERANCE = 0.5

increasing = None

dut_amplifier.set_gain("dl", 0); low_reg_output = analyzer.get_level_from_marker(1)
dut_amplifier.set_gain("dl", 4095); high_reg_output = analyzer.get_level_from_marker(1)

increasing = high_reg_output >low_reg_output

# Factibility check
low_value_check, high_value_check = sorted((low_reg_output, high_reg_output))
if not(low_value_check - TOLERANCE <= TARGET_PW <= high_value_check + TOLERANCE):
    print("DUT out of possible ranges")

best = None

low = 0
high = 4095

while low <= high:
    mid_value = (low + high) // 2
    print(mid_value)
    dut_amplifier.set_gain("DL", mid_value)
    current_power_out = analyzer.get_level_from_marker(1)
    current_error = abs(current_power_out - TARGET_PW)
    if best is None or current_error < best[0]:
        best = (current_error, mid_value, current_power_out)
    if current_error <= TOLERANCE:
        print(f"REG_VAL: {mid_value} | PW OUT: {current_power_out}")
        break
    if(current_power_out < TARGET_PW) == increasing:
        low = mid_value + 1
    else:
        high = mid_value -1

if best[0] <= TOLERANCE:
    print(f"best-0 : {best[1]} | best-1 : {best[2]}")


2047
REG_VAL: 2047 | PW OUT: -26.18
best-0 : 2047 | best-1 : -26.18
